# 🧪 Conduit API (`https://conduit.ozdoev.net/v1`) — Live Verification & Testing Notebook

This standalone Jupyter Notebook allows you to instantly verify your **Conduit API Key** and test model inferences with OpenAI models (such as `openai/gpt-5.5-preview`, `openai/gpt-5.5`, or `gpt-4o`) before launching the full **AlphaVision AI** application.

### 📖 Instructions:
1. Paste your Conduit API key into the `CONDUIT_API_KEY` variable in **Cell 1**.
2. Run **Cell 1** to initialize credentials.
3. Run **Cell 2** to verify standard conversational financial analytical chat.
4. Run **Cell 3** to verify structured JSON output generation (the required format used by our news & sentiment engine).

In [18]:
import os
import json
import urllib.request
import urllib.error

# ======================================================================
# 1. PASTE YOUR CONDUIT API KEY BELOW:
# ======================================================================
CONDUIT_API_KEY = "sk-cdt-eyJpZCI6Ijc5MDE0ODE5NDciLCJ1IjoiIiwibiI6ImRlZmF1bHQiLCJqIjoiZGVmYXVsdCIsImsiOiJhcGkifQ.VVzJ2nG08S9uzsJTu0u7-ibdEKvMnsqt1hC-ToRp7pk"
CONDUIT_BASE_URL = "https://conduit.ozdoev.net/v1"

# ======================================================================
# 2. SELECT MODEL TO TEST (e.g., openai/gpt-5.5-preview, gpt-4o, etc.)
# ======================================================================
MODEL_NAME = "deepseek-v3.2"

print("=========================================================================")
print(f"🔒 API Key Status: {'CONFIGURED ✅' if CONDUIT_API_KEY != 'your_conduit_api_key_here' else 'PLEASE PASTE YOUR KEY ABOVE ❌'}")
print(f"🌐 Target Gateway: {CONDUIT_BASE_URL}")
print(f"🤖 Selected Model: {MODEL_NAME}")
print("=========================================================================")

🔒 API Key Status: CONFIGURED ✅
🌐 Target Gateway: https://conduit.ozdoev.net/v1
🤖 Selected Model: deepseek-v3.2


In [27]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-cdt-eyJpZCI6Ijc5MDE0ODE5NDciLCJ1IjoiIiwibiI6ImRlZmF1bHQiLCJqIjoiZGVmYXVsdCIsImsiOiJhcGkifQ.VVzJ2nG08S9uzsJTu0u7-ibdEKvMnsqt1hC-ToRp7pk",
    base_url = "https://conduit.ozdoev.net/v1"
)

models = client.models.list()

for model in models.data:
    print(model.id)

opus-5-preview
claude-sonnet-4-6
claude-haiku-4-5
claude-4-8
claude-fable-5
claude-fable-5-1m
claude-opus-4-8-latest
claude-opus-4.8
codestral
deepseek-r1
deepseek-v3.2
deepseek-v4-flash
fable
fable-1m
fable-5
fable[1m]
gemini-2.5-flash
gemini-2.5-pro
gemini-3-pro
gpt-4.1
gpt-4o
gpt-5
gpt-5-mini
gpt-5.6
gpt-5.6-sol
gpt-5.6-sol-extra-high
grok-3
grok-3-mini
grok-4
kimi-k2.6
llama-3.3-70b
llama-4-maverick
llama-4-scout
mistral-large-3
mixtral
o4
opus
qwen3-coder
qwen3-max


In [25]:
try:
    response = client.chat.completions.create(
        model="llama-4-scout",
        messages=[
            {"role": "user", "content": "Hello!"}
        ]
    )
    print(response)
except Exception as e:
    print(type(e))
    print(e)

<class 'openai.InternalServerError'>
Error code: 503 - {'error': {'message': 'Temporary service interruption. Retry the last turn; your conversation and tool state are preserved.', 'type': 'server_error'}}


In [19]:
# ======================================================================
# TEST 1: Standard Conversational Financial & Econometric Inquiry
# ======================================================================
payload_chat = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "system", 
            "content": "You are AlphaVision AI, an expert institutional quantitative equities analyst powered by Conduit."
        },
        {
            "role": "user", 
            "content": "In 3 bullet points, explain why combining ARIMA trend forecasting with GARCH conditional volatility modeling creates a reliable trading corridor for stock investors."
        }
    ],
    "temperature": 0.3,
    "max_tokens": 350
}

headers = {
    "Authorization": f"Bearer {CONDUIT_API_KEY}",
    "Content-Type": "application/json",
    "User-Agent": "AlphaVision-AI-Test-Client/1.0"
}

print(f"📡 Transmitting request to {MODEL_NAME} via Conduit API Gateway...")
try:
    req = urllib.request.Request(CONDUIT_BASE_URL, data=json.dumps(payload_chat).encode('utf-8'), headers=headers)
    with urllib.request.urlopen(req, timeout=35) as resp:
        response_data = json.loads(resp.read().decode('utf-8'))
        generated_text = response_data['choices'][0]['message']['content']
        print("\n✅ CONDUIT API TEST 1 SUCCESSFUL! Model Response:\n" + "="*75)
        print(generated_text)
        print("="*75)
except urllib.error.HTTPError as e:
    print(f"\n❌ HTTP Error {e.code}: {e.reason}")
    print("Server details:", e.read().decode('utf-8', errors='ignore'))
except Exception as e:
    print(f"\n❌ Connection Error: {e}")

📡 Transmitting request to deepseek-v3.2 via Conduit API Gateway...

❌ HTTP Error 404: Not Found
Server details: {"type":"error","error":{"type":"not_found_error","message":"Unknown endpoint: /v1"}}


In [ ]:
# ======================================================================
# TEST 2: Structured JSON Financial Sentiment Pipeline Verification
# ======================================================================
# This tests whether the endpoint supports strictly formatted JSON output,
# which AlphaVision AI uses to quantify real-time financial news sentiment.

payload_json = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "system", 
            "content": "You are an institutional quantitative financial NLP engine. Output ONLY valid JSON matching the requested structure without markdown code blocks or explanatory chatter."
        },
        {
            "role": "user", 
            "content": "Analyze this headline: 'NVIDIA data center computing revenue surges 210%, exceeding institutional estimates; management announces new $15B share buyback program.' Output a single JSON object with exact keys: \"ticker\", \"classification\" (Bullish, Bearish, or Neutral), \"sentiment_score\" (from -100 to 100), and \"key_catalyst\"."
        }
    ],
    "temperature": 0.1,
    "response_format": {"type": "json_object"}
}

print(f"📡 Testing Structured JSON-Mode via Conduit API...")
try:
    req_json = urllib.request.Request(CONDUIT_BASE_URL, data=json.dumps(payload_json).encode('utf-8'), headers=headers)
    with urllib.request.urlopen(req_json, timeout=35) as resp:
        response_data = json.loads(resp.read().decode('utf-8'))
        raw_content = response_data['choices'][0]['message']['content'].strip()
        
        # Strip markdown syntax if accidentally included by LLM
        if raw_content.startswith("```json"):
            raw_content = raw_content[7:-3].strip()
        elif raw_content.startswith("```"):
            raw_content = raw_content[3:-3].strip()
            
        parsed_json = json.loads(raw_content)
        print("\n✅ CONDUIT JSON PIPELINE TEST SUCCESSFUL! Parsed JSON Output:\n" + "="*75)
        print(json.dumps(parsed_json, indent=2))
        print("="*75)
except urllib.error.HTTPError as e:
    print(f"\n❌ HTTP Error {e.code}: {e.reason}")
    print("Server details:", e.read().decode('utf-8', errors='ignore'))
except Exception as e:
    print(f"\n❌ JSON Execution Error: {e}")